# PaddleOCR Full-Page Prescription Training — Google Colab

This notebook trains **both Detection + Recognition** models on **full prescription images**
(not cropped text lines).

**What's different from the line-level notebook:**
- Generates full-page prescription images with **bounding box annotations**
- Fine-tunes the **Detection model (DB++)** to find text regions on prescriptions
- Fine-tunes the **Recognition model (SVTR_LCNet)** on cropped text from those annotations
- Tests end-to-end on full prescription pages

**Steps:**
1. Install PaddlePaddle GPU + PaddleOCR
2. Clone PaddleOCR repo (training tools)
3. Generate synthetic full-page prescriptions with bounding box annotations
4. Prepare detection + recognition training data
5. Train detection model (DB++)
6. Train recognition model (SVTR_LCNet)
7. Export & test both models end-to-end
8. Download for local deployment

> **Runtime:** Go to `Runtime > Change runtime type > GPU (T4)` before running!

## 1. Setup Environment

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# === INSTALL EVERYTHING (one cell, one session restart) ===
import subprocess, sys

def run(cmd):
    print(f"\n{'='*60}\n>>> {cmd}\n{'='*60}")
    subprocess.check_call(cmd, shell=True)

# 1. Pin numpy 1.x FIRST
run('pip install "numpy==1.26.4" --force-reinstall --quiet')

# 2. Install PaddlePaddle GPU
run('pip install paddlepaddle-gpu==2.6.2 -i https://pypi.tuna.tsinghua.edu.cn/simple --quiet')

# 3. Install PaddleOCR
run('pip install paddleocr==2.7.3 --quiet')

# 4. Reinstall C extension packages against numpy 1.x
run('pip install opencv-python opencv-python-headless shapely imgaug --force-reinstall --no-cache-dir --quiet')

# 5. Pin Pillow to a compatible version (10.x works; 11+ has _Ink import issues on Colab)
run('pip install "Pillow>=10.0.0,<11.0.0" PyYAML --force-reinstall --quiet')

# 6. Final numpy pin
run('pip install "numpy==1.26.4" --force-reinstall --quiet')

print("\n" + "=="*30)
print("ALL DONE! Now go to: Runtime -> Restart session")
print("Then run the VERIFY cell below.")
print("=="*30)

In [ ]:
# === VERIFY (run AFTER session restart) ===
import numpy as np
print(f"NumPy: {np.__version__}")
v = int(np.__version__.split('.')[0])
if v >= 2:
    print("ERROR: numpy is still 2.x! Re-run the install cell above, then restart session again.")
else:
    print("NumPy 1.x -- good!")

import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.device.is_compiled_with_cuda()}")
if paddle.device.is_compiled_with_cuda():
    print(f"GPU count: {paddle.device.cuda.device_count()}")

import cv2
print(f"OpenCV: {cv2.__version__}")

from PIL import Image
print(f"Pillow: {Image.__version__}")

print("\nAll imports OK. Continue with the next cell.")

## 2. Clone PaddleOCR Repository

In [ ]:
import os
WORK_DIR = '/content/prescription-ocr-fullpage'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Clone PaddleOCR repo (training tools)
if not os.path.exists('PaddleOCR'):
    !git clone --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
    !pip install -r PaddleOCR/requirements.txt --quiet
else:
    print("PaddleOCR repo already cloned")

## 3. Generate Full-Page Training Data with Bounding Boxes

This generates **full prescription page images** with precise bounding box annotations for every text line.
The bounding boxes are used to train the **detection model**, and the cropped regions train the **recognition model**.

In [ ]:
import random
import json
from pathlib import Path
from datetime import datetime, timedelta
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import numpy as np

# Configuration
NUM_PRESCRIPTIONS = 600
PAGE_WIDTH = 800
PAGE_HEIGHT = 1100
QUALITY_WEIGHTS = [0.50, 0.30, 0.20]  # high, medium, low

# ==================== PRESCRIPTION DATA ====================

PATIENT_NAMES = [
    "John Silva", "Maria Perera", "Kasun Fernando", "Dilini Rajapaksa",
    "Nimal De Silva", "Sanduni Wickramasinghe", "Chaminda Jayawardena",
    "Rushika Gunasekara", "Thilini Dissanayake", "Nuwan Bandara",
    "Hasini Samaraweera", "Rohan Gamage", "Savithri Herath",
    "Prasanna Mendis", "Ayesha Pathirana", "Lakmal Rathnayake",
    "Ishara Rodrigo", "Buddhika Amarasinghe", "Shanika Kumarasinghe",
    "Malith Jayakody", "Nadeesha Wickremasinghe", "Udara Gamage", "Yasmin Fonseka",
    "Sameera Pathirana", "Janaki Munasinghe", "Kusal Rajapakse", "Dilrukshi Senanayake",
    "Asanka Herath", "Minoli Samarasinghe", "Gayan Liyanage", "Rashmi Wanninayake",
    "Manoj Gunathilake", "Thanuja Ratnayake", "Sachini Dharmasena", "Buddhika Wijesekara",
    "Hasini Kudagama", "Danushka Perera", "Amaya Cooray", "Upul Mendis",
    "Nethmi Karunaratne", "Charith Atapattu", "Madushani Ekanayake", "Dinesh Madushanka",
    "Shashika Priyankara", "Chaminda Rajakaruna", "Gayani Kapukotuwa", "Nuwan Pradeep",
    "Tharanga Vithanage", "Senali Weerasinghe", "Kavindya Gunaratne", "Chatura Alwis",
    "Dimuthu Karunaratne", "Niluka Fernando", "Prabath Nissanka", "Shalini Kodagoda",
    "Anuradha Jayalath", "Hemantha Silva", "Priyani Wijesinghe", "Sanjeewa Cooray",
    "Rajesh Kumar", "Priya Sharma", "Amit Patel", "Sneha Reddy", "Vikram Singh",
    "Anjali Gupta", "Suresh Nair", "Kavita Menon", "Arjun Krishnan", "Deepa Iyer",
    "David Anderson", "Sarah Johnson", "Michael Chen", "Emily Williams", "James Brown",
    "Lisa Davis", "Robert Martinez", "Jennifer Garcia", "William Rodriguez", "Mary Wilson",
    "Ahmed Hassan", "Fatima Ali", "Omar Abdullah", "Aisha Khan", "Mohammed Rahman",
    "Li Wei", "Zhang Mei", "Wang Jian", "Tanaka Hiroshi", "Kim Min-Jun"
]

DOCTOR_NAMES = [
    "Dr. A. Fernando", "Dr. K. Perera", "Dr. S. Silva",
    "Dr. M. Gunawardena", "Dr. R. Jayasinghe", "Dr. N. Dissanayake",
    "Dr. P. Wickramasinghe", "Dr. T. Ratnayake",
    "Dr. Chandana Wijesuriya", "Dr. Lakshmi Fernando", "Dr. Ajith Perera",
    "Dr. Suresh Bandara", "Dr. Nirmala Silva", "Dr. Rohan Jayasuriya",
    "Dr. Priyanka De Silva", "Dr. Mahesh Rathnayake", "Dr. Sampath Kumarasinghe",
    "Dr. Anoma Jayawardena", "Dr. Nishan Gunaratne", "Dr. Buddhika Wickramasinghe",
    "Dr. Chamari Amarasena", "Dr. Ruwan Samarakoon", "Dr. Manjula Seneviratne",
    "Dr. Sarah Chen", "Dr. Michael Thompson", "Dr. Priya Sharma",
    "Dr. David Anderson", "Dr. Emily Williams", "Dr. Ahmed Hassan"
]

HOSPITALS = [
    "Lanka Hospital", "Nawaloka Hospital", "Asiri Medical Center",
    "Durdans Hospital", "Apollo Hospital", "Central Hospital",
    "City Medical Center", "National Hospital",
    "Asiri Central Hospital", "Hemas Hospital", "Oasis Hospital",
    "Ninewells Hospital", "Golden Key Hospital", "Royal Hospital",
    "National Hospital of Sri Lanka", "Lady Ridgeway Hospital",
    "General Hospital Kandy", "Teaching Hospital Karapitiya",
    "St. Michael's Medical Center", "Metro Healthcare Hospital",
    "Prime Care Hospital", "Sunshine Medical Center", "Greenfield Hospital",
    "Riverside Medical Center", "Valley View Hospital", "Summit Healthcare",
    "Lakeside Medical Center", "Harbor Medical Hospital", "Clearwater Hospital"
]

MEDICINES = [
    {"name": "Paracetamol", "dosages": ["500mg", "650mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Amoxicillin", "dosages": ["250mg", "500mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Omeprazole", "dosages": ["20mg", "40mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Metformin", "dosages": ["500mg", "1000mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Amlodipine", "dosages": ["5mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Atorvastatin", "dosages": ["10mg", "20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Aspirin", "dosages": ["75mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Cetirizine", "dosages": ["10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Azithromycin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Losartan", "dosages": ["25mg", "50mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Vitamin D3", "dosages": ["1000IU", "2000IU"], "forms": ["Capsule", "Cap"]},
    {"name": "Folic Acid", "dosages": ["5mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Ibuprofen", "dosages": ["200mg", "400mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Ciprofloxacin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Doxycycline", "dosages": ["100mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Simvastatin", "dosages": ["10mg", "20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Lisinopril", "dosages": ["5mg", "10mg", "20mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Levothyroxine", "dosages": ["50mcg", "100mcg"], "forms": ["Tablet", "Tab"]},
    {"name": "Pantoprazole", "dosages": ["20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Diclofenac", "dosages": ["25mg", "50mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Gabapentin", "dosages": ["100mg", "300mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Furosemide", "dosages": ["20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Warfarin", "dosages": ["2mg", "5mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Propranolol", "dosages": ["10mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Clarithromycin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Montelukast", "dosages": ["4mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Clopidogrel", "dosages": ["75mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Rosuvastatin", "dosages": ["5mg", "10mg", "20mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Fexofenadine", "dosages": ["120mg", "180mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Metoprolol", "dosages": ["25mg", "50mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Sertraline", "dosages": ["50mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Prednisolone", "dosages": ["5mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Salbutamol", "dosages": ["2mg", "4mg"], "forms": ["Tablet", "Inhaler"]},
    {"name": "Calcium Carbonate", "dosages": ["500mg", "1000mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Tramadol", "dosages": ["50mg", "100mg"], "forms": ["Tablet", "Tab"]}
]

FREQUENCIES = [
    "1-0-1 (Twice daily)", "1-1-1 (Three times daily)",
    "0-0-1 (Once daily at night)", "1-0-0 (Once daily in morning)",
    "Twice daily", "Three times daily", "As needed",
    "Every 6 hours", "Every 8 hours", "Every 12 hours",
    "At bedtime", "Before meals", "After meals"
]

DURATIONS = [
    "5 days", "7 days", "10 days", "14 days", "1 month",
    "3 months", "3 days", "21 days", "2 weeks", "30 days"
]

print("Data definitions loaded!")
print(f"  Patients: {len(PATIENT_NAMES)}, Doctors: {len(DOCTOR_NAMES)}")
print(f"  Hospitals: {len(HOSPITALS)}, Medicines: {len(MEDICINES)}")

In [ ]:
# Font loading (Colab uses Linux fonts)
def load_fonts():
    font_families = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
    ]
    os.system('apt-get install -y fonts-liberation fonts-dejavu fonts-freefont-ttf fonts-ubuntu > /dev/null 2>&1')
    extra = [
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSerif.ttf",
        "/usr/share/fonts/truetype/freefont/FreeMono.ttf",
        "/usr/share/fonts/truetype/ubuntu/Ubuntu-R.ttf",
        "/usr/share/fonts/truetype/ubuntu/UbuntuMono-R.ttf",
    ]
    font_families.extend(extra)

    sizes = {'title': 32, 'header': 26, 'normal': 22, 'small': 18, 'tiny': 15}
    available_fonts = []
    for font_path in font_families:
        try:
            ImageFont.truetype(font_path, 24)
            available_fonts.append(font_path)
        except OSError:
            continue

    if not available_fonts:
        print("WARNING: No fonts found, using default")
        return [{key: ImageFont.load_default() for key in sizes}]

    print(f"Found {len(available_fonts)} fonts")
    all_font_sets = []
    for font_name in available_fonts:
        font_set = {}
        for key, size in sizes.items():
            try:
                font_set[key] = ImageFont.truetype(font_name, size)
            except OSError:
                font_set[key] = ImageFont.load_default()
        all_font_sets.append(font_set)
    return all_font_sets


def random_date():
    days_ago = random.randint(0, 30)
    d = datetime.now() - timedelta(days=days_ago)
    fmt = random.choice(["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"])
    return d.strftime(fmt)


def degrade_image(img, quality):
    """Apply quality degradation to full-page image"""
    if quality == 'high':
        if random.random() > 0.5:
            factor = random.uniform(0.9, 1.1)
            img = ImageEnhance.Brightness(img).enhance(factor)
        return img
    if quality == 'medium':
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
        angle = random.uniform(-1.5, 1.5)
        img = img.rotate(angle, fillcolor='white', expand=False)
        arr = np.array(img)
        noise = np.random.normal(0, 4, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)
    # low quality
    img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(1.0, 2.5)))
    angle = random.uniform(-3, 3)
    img = img.rotate(angle, fillcolor='white', expand=False)
    arr = np.array(img)
    noise = np.random.normal(0, 10, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def pick_quality():
    return random.choices(['high', 'medium', 'low'], weights=QUALITY_WEIGHTS)[0]


def get_text_bbox(draw, text, font):
    """Get text bounding box dimensions"""
    bbox = draw.textbbox((0, 0), text, font=font)
    return bbox[2] - bbox[0], bbox[3] - bbox[1]  # width, height


print("Helper functions defined!")

In [ ]:
# ==================== FULL-PAGE PRESCRIPTION RENDERER ====================

def render_full_prescription(rx_data, fonts, page_w=PAGE_WIDTH, page_h=PAGE_HEIGHT):
    """
    Render a full prescription page image.
    Returns: (PIL Image, list of {bbox: [[x1,y1],[x2,y1],[x2,y2],[x1,y2]], text: str})
    """
    img = Image.new('RGB', (page_w, page_h), 'white')
    draw = ImageDraw.Draw(img)
    annotations = []

    x_margin = random.randint(30, 60)
    y = random.randint(20, 40)
    line_spacing = random.randint(6, 12)

    def draw_text_line(text, font_key, x_offset=None, center=False):
        nonlocal y
        font = fonts[font_key]
        tw, th = get_text_bbox(draw, text, font)

        if center:
            x = (page_w - tw) // 2
        elif x_offset is not None:
            x = x_offset
        else:
            x = x_margin

        x += random.randint(-2, 2)
        y += random.randint(-1, 1)

        draw.text((x, y), text, fill='black', font=font)

        x1, y1 = x - 1, y - 1
        x2, y2 = x + tw + 1, y + th + 1
        annotations.append({
            'bbox': [[x1, y1], [x2, y1], [x2, y2], [x1, y2]],
            'text': text
        })

        y += th + line_spacing
        return th

    # ---- Header ----
    header_style = random.choice(['style1', 'style2', 'style3'])

    if header_style == 'style1':
        draw_text_line(rx_data['hospital'], 'title', center=True)
        y += 5
        draw_text_line(f"Patient Name: {rx_data['patient']}", 'normal')
        draw_text_line(f"Age: {rx_data['age']} years", 'normal')
        draw_text_line(f"Date: {rx_data['date']}", 'normal')
        draw_text_line(f"Doctor: {rx_data['doctor']}", 'normal')
    elif header_style == 'style2':
        draw_text_line(rx_data['hospital'], 'title', center=True)
        draw_text_line(rx_data['doctor'], 'header', center=True)
        y += 8
        draw_text_line(f"Patient: {rx_data['patient']}", 'normal')
        draw_text_line(f"Age: {rx_data['age']}    Date: {rx_data['date']}", 'normal')
    else:
        draw_text_line(rx_data['hospital'], 'header', center=True)
        y += 4
        draw_text_line(f"Doctor: {rx_data['doctor']}", 'normal')
        draw_text_line(f"Patient Name: {rx_data['patient']}   Age: {rx_data['age']}", 'normal')
        draw_text_line(f"Date: {rx_data['date']}", 'normal')

    # ---- Separator ----
    y += random.randint(5, 15)
    sep_style = random.choice(['rx', 'line', 'both'])
    if sep_style in ('line', 'both'):
        draw.line([(x_margin, y), (page_w - x_margin, y)], fill='black', width=1)
        y += 10
    if sep_style in ('rx', 'both'):
        draw_text_line("Rx", 'header')
    y += 5

    # ---- Medications ----
    med_format = rx_data['format']
    for mi, med_info in enumerate(rx_data['medications'], 1):
        med_name = med_info['name']
        dosage = med_info['dosage']
        form = med_info['form']
        freq = med_info['frequency']
        dur = med_info['duration']

        if med_format == 'A':
            draw_text_line(f"{mi}. {med_name} {dosage} {form}", 'normal')
            draw_text_line(f"   {freq}", 'small', x_offset=x_margin + 20)
            draw_text_line(f"   Duration: {dur}", 'small', x_offset=x_margin + 20)
        elif med_format == 'B':
            draw_text_line(f"{mi}. {med_name}", 'normal')
            draw_text_line(f"   Dose: {dosage} {form}", 'small', x_offset=x_margin + 20)
            draw_text_line(f"   {freq}", 'small', x_offset=x_margin + 20)
            draw_text_line(f"   Duration: {dur}", 'small', x_offset=x_margin + 20)
        else:
            draw_text_line(med_name, 'normal')
            draw_text_line(f"Dose: {dosage} {form}", 'small', x_offset=x_margin + 15)
            draw_text_line(freq, 'small', x_offset=x_margin + 15)
            draw_text_line(f"Duration: {dur}", 'small', x_offset=x_margin + 15)

        y += random.randint(3, 8)

    # ---- Footer ----
    y += random.randint(15, 30)
    footer_style = random.choice(['sig', 'sig_date', 'full'])
    if footer_style == 'sig':
        draw_text_line(f"Signature: {rx_data['doctor']}", 'normal')
    elif footer_style == 'sig_date':
        draw_text_line(f"Signature: {rx_data['doctor']}", 'normal')
        draw_text_line(f"Date: {rx_data['date']}", 'small')
    else:
        draw_text_line(f"Prescribed by: {rx_data['doctor']}", 'normal')
        draw_text_line(f"Registration No: {random.randint(10000,99999)}", 'small')
        draw_text_line(f"Date: {rx_data['date']}", 'small')

    return img, annotations


print("Full-page renderer defined!")

In [ ]:
# ==================== GENERATE FULL-PAGE DATASET ====================

TRAINING_DATA_DIR = Path(WORK_DIR) / 'training_data'
det_train_dir = TRAINING_DATA_DIR / 'det' / 'train'
det_val_dir = TRAINING_DATA_DIR / 'det' / 'val'
rec_train_dir = TRAINING_DATA_DIR / 'rec' / 'train'
rec_val_dir = TRAINING_DATA_DIR / 'rec' / 'val'

for d in [det_train_dir, det_val_dir, rec_train_dir, rec_val_dir]:
    if d.exists():
        for f in d.glob('*.png'):
            f.unlink()
    d.mkdir(parents=True, exist_ok=True)

font_sets = load_fonts()

det_labels = []
rec_samples = []
quality_counts = {'high': 0, 'medium': 0, 'low': 0}
total_crops = 0

print(f"\nGenerating {NUM_PRESCRIPTIONS} full-page prescriptions...")

for rx_idx in range(1, NUM_PRESCRIPTIONS + 1):
    num_meds = random.randint(1, 5)
    meds = []
    for _ in range(num_meds):
        med = random.choice(MEDICINES)
        meds.append({
            'name': med['name'],
            'dosage': random.choice(med['dosages']),
            'form': random.choice(med['forms']),
            'frequency': random.choice(FREQUENCIES),
            'duration': random.choice(DURATIONS),
        })

    rx_data = {
        'patient': random.choice(PATIENT_NAMES),
        'doctor': random.choice(DOCTOR_NAMES),
        'hospital': random.choice(HOSPITALS),
        'date': random_date(),
        'age': random.randint(18, 80),
        'medications': meds,
        'format': random.choice(['A', 'B', 'C']),
    }

    quality = pick_quality()
    quality_counts[quality] += 1
    fonts = random.choice(font_sets)

    page_img, annotations = render_full_prescription(rx_data, fonts)
    page_img = degrade_image(page_img, quality)

    page_filename = f"rx_{rx_idx:04d}_{rx_data['format']}_{quality}.png"
    page_img.save(det_train_dir / page_filename)

    det_annots = []
    for ann in annotations:
        det_annots.append({
            'transcription': ann['text'],
            'points': ann['bbox'],
        })
    det_labels.append((f"train/{page_filename}", json.dumps(det_annots, ensure_ascii=False)))

    page_arr = np.array(page_img)
    for ann_idx, ann in enumerate(annotations):
        bbox = ann['bbox']
        x1 = max(0, int(bbox[0][0]))
        y1 = max(0, int(bbox[0][1]))
        x2 = min(page_arr.shape[1], int(bbox[2][0]))
        y2 = min(page_arr.shape[0], int(bbox[2][1]))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = page_arr[y1:y2, x1:x2]
        crop_img = Image.fromarray(crop)

        target_h = 48
        ratio = target_h / crop_img.height
        target_w = max(10, int(crop_img.width * ratio))
        crop_img = crop_img.resize((target_w, target_h), Image.Resampling.LANCZOS)

        total_crops += 1
        crop_filename = f"crop_{rx_idx:04d}_{ann_idx:03d}.png"
        crop_img.save(rec_train_dir / crop_filename)
        rec_samples.append((crop_filename, ann['text'].strip()))

    if rx_idx % 100 == 0:
        print(f"  {rx_idx}/{NUM_PRESCRIPTIONS} pages ({total_crops} crops)")

print(f"\nGenerated {NUM_PRESCRIPTIONS} full-page images, {total_crops} text crops")
print(f"Quality distribution: {quality_counts}")

In [ ]:
# ==================== SPLIT DATA & WRITE LABEL FILES ====================

random.seed(42)

# ----- Detection split (90/10) -----
random.shuffle(det_labels)
det_split = int(0.9 * len(det_labels))
det_train_labels = det_labels[:det_split]
det_val_labels = det_labels[det_split:]

for img_path, _ in det_val_labels:
    src = det_train_dir / Path(img_path).name
    dst = det_val_dir / Path(img_path).name
    if src.exists():
        src.rename(dst)

det_data_dir = TRAINING_DATA_DIR / 'det'
with open(det_data_dir / 'train_label.txt', 'w', encoding='utf-8') as f:
    for img_path, annots_json in det_train_labels:
        f.write(f"{img_path}\t{annots_json}\n")

with open(det_data_dir / 'val_label.txt', 'w', encoding='utf-8') as f:
    for img_path, annots_json in det_val_labels:
        path = img_path.replace('train/', 'val/')
        f.write(f"{path}\t{annots_json}\n")

print(f"Detection: {len(det_train_labels)} train, {len(det_val_labels)} val")

# ----- Recognition split (90/10) -----
random.shuffle(rec_samples)
rec_split = int(0.9 * len(rec_samples))
rec_train_samples = rec_samples[:rec_split]
rec_val_samples = rec_samples[rec_split:]

for filename, _ in rec_val_samples:
    src = rec_train_dir / filename
    dst = rec_val_dir / filename
    if src.exists():
        src.rename(dst)

rec_data_dir = TRAINING_DATA_DIR / 'rec'
with open(rec_data_dir / 'train_label.txt', 'w', encoding='utf-8') as f:
    for filename, label in rec_train_samples:
        f.write(f"train/{filename}\t{label}\n")

with open(rec_data_dir / 'val_label.txt', 'w', encoding='utf-8') as f:
    for filename, label in rec_val_samples:
        f.write(f"val/{filename}\t{label}\n")

# Character dictionary
all_chars = set()
for _, label in rec_samples:
    all_chars.update(label)
dict_file = TRAINING_DATA_DIR / 'en_dict.txt'
with open(dict_file, 'w', encoding='utf-8') as f:
    for ch in sorted(all_chars):
        f.write(f"{ch}\n")

print(f"Recognition: {len(rec_train_samples)} train, {len(rec_val_samples)} val")
print(f"Dictionary: {len(all_chars)} characters")
print(f"All label files written to {TRAINING_DATA_DIR}")

In [ ]:
# Preview full-page training samples with bounding boxes
import matplotlib.pyplot as plt
import matplotlib.patches as patches

sample_files = sorted(det_train_dir.glob('*.png'))[:3]
fig, axes = plt.subplots(1, len(sample_files), figsize=(18, 12))
if len(sample_files) == 1:
    axes = [axes]

label_dict = {}
with open(det_data_dir / 'train_label.txt', 'r') as f:
    for line in f:
        parts = line.strip().split('\t', 1)
        label_dict[Path(parts[0]).name] = json.loads(parts[1])

for ax, img_path in zip(axes, sample_files):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    if img_path.name in label_dict:
        for ann in label_dict[img_path.name]:
            pts = ann['points']
            x1, y1 = pts[0]
            w = pts[2][0] - pts[0][0]
            h = pts[2][1] - pts[0][1]
            rect = patches.Rectangle((x1, y1), w, h,
                                     linewidth=1, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
    ax.axis('off')

plt.suptitle('Full-Page Prescriptions with Detection Bounding Boxes', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Download Pretrained Models

We need TWO pretrained models:
- **Detection**: `en_PP-OCRv4_det` - text region detection
- **Recognition**: `en_PP-OCRv4_rec` - text recognition

In [ ]:
PRETRAINED_DIR = Path(WORK_DIR) / 'pretrained'
PRETRAINED_DIR.mkdir(exist_ok=True)

def find_pretrained_checkpoint(model_dir):
    """Auto-detect the checkpoint name inside a pretrained model directory.
    PP-OCRv4 archives may contain best_accuracy.pdparams, best_model.pdparams,
    student.pdparams, or other names depending on the version."""
    for name in ['best_accuracy', 'best_model', 'student']:
        if (model_dir / f'{name}.pdparams').exists():
            return str(model_dir / name)
    # Fallback: any .pdparams file
    params = sorted(model_dir.glob('*.pdparams'))
    if params:
        return str(params[0]).replace('.pdparams', '')
    raise FileNotFoundError(f"No .pdparams found in {model_dir}. Contents: {list(model_dir.iterdir())}")

def download_and_extract(url, dest_dir, model_name):
    """Download a pretrained model tar, extract it, and clean up."""
    import subprocess
    tar_path = f"{dest_dir}/{model_name}.tar"
    print(f"Downloading {url} ...")
    result = subprocess.run(
        ["wget", "-q", "--no-check-certificate", "-O", tar_path, url],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"wget failed: {result.stderr}")
        raise RuntimeError(f"Failed to download {url}")

    # Verify it's actually a tar file
    result = subprocess.run(["file", tar_path], capture_output=True, text=True)
    print(f"File type: {result.stdout.strip()}")

    result = subprocess.run(
        ["tar", "-xf", tar_path, "-C", str(dest_dir)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"tar extraction failed: {result.stderr}")
        # Try as gzip
        result = subprocess.run(
            ["tar", "-xzf", tar_path, "-C", str(dest_dir)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f"Cannot extract {tar_path}")

    os.remove(tar_path)
    print(f"Extracted to {dest_dir}")

# ---- Detection pretrained model ----
# NOTE: PP-OCRv4 detection is language-agnostic (same model for all languages).
# There is no en_PP-OCRv4_det — use the ch (Chinese) det model which works for English too.
DET_MODEL_NAME = 'ch_PP-OCRv4_det_train'
DET_PRETRAINED_URL = f'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/{DET_MODEL_NAME}.tar'
det_model_dir = PRETRAINED_DIR / DET_MODEL_NAME

if not det_model_dir.exists():
    download_and_extract(DET_PRETRAINED_URL, PRETRAINED_DIR, DET_MODEL_NAME)
    # The tar might extract into a subdirectory with a different name; find it
    if not det_model_dir.exists():
        extracted = [d for d in PRETRAINED_DIR.iterdir() if d.is_dir() and 'det' in d.name.lower()]
        if extracted:
            det_model_dir = extracted[0]
            print(f"Detected extraction dir: {det_model_dir}")
    print(f"Downloaded detection model to {det_model_dir}")
else:
    print(f"Detection model exists at {det_model_dir}")

print("Detection model files:")
for f in sorted(det_model_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

det_pretrained_path = find_pretrained_checkpoint(det_model_dir)
print(f"Detection checkpoint: {det_pretrained_path}")

# ---- Recognition pretrained model ----
REC_MODEL_NAME = 'en_PP-OCRv4_rec_train'
REC_PRETRAINED_URL = f'https://paddleocr.bj.bcebos.com/PP-OCRv4/english/{REC_MODEL_NAME}.tar'
rec_model_dir = PRETRAINED_DIR / REC_MODEL_NAME

if not rec_model_dir.exists():
    download_and_extract(REC_PRETRAINED_URL, PRETRAINED_DIR, REC_MODEL_NAME)
    if not rec_model_dir.exists():
        extracted = [d for d in PRETRAINED_DIR.iterdir() if d.is_dir() and 'rec' in d.name.lower()]
        if extracted:
            rec_model_dir = extracted[0]
            print(f"Detected extraction dir: {rec_model_dir}")
    print(f"Downloaded recognition model to {rec_model_dir}")
else:
    print(f"Recognition model exists at {rec_model_dir}")

print("Recognition model files:")
for f in sorted(rec_model_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

rec_pretrained_path = find_pretrained_checkpoint(rec_model_dir)
print(f"Recognition checkpoint: {rec_pretrained_path}")

## 5. Train Detection Model (DB++)

Fine-tunes the PP-OCRv4 detection model on full prescription page images.
This teaches the model **where text is located** on prescription pages.

In [ ]:
import yaml

DET_EPOCHS = 100
DET_BATCH_SIZE = 8
DET_LEARNING_RATE = 0.0001

DET_OUTPUT_DIR = Path(WORK_DIR) / 'model' / 'det'
DET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

det_config = {
    'Global': {
        'debug': False,
        'use_gpu': True,
        'epoch_num': DET_EPOCHS,
        'log_smooth_window': 20,
        'print_batch_step': 10,
        'save_model_dir': str(DET_OUTPUT_DIR / 'training_output'),
        'save_epoch_step': 20,
        'eval_batch_step': [0, 500],
        'cal_metric_during_train': True,
        'pretrained_model': det_pretrained_path,
        'checkpoints': None,
        'save_inference_dir': str(DET_OUTPUT_DIR / 'inference'),
        'use_visualdl': False,
        'infer_img': None,
        'infer_mode': False,
        'distributed': False,
        'save_res_path': str(DET_OUTPUT_DIR / 'det_results.txt'),
    },
    'Optimizer': {
        'name': 'Adam',
        'beta1': 0.9,
        'beta2': 0.999,
        'lr': {
            'name': 'Cosine',
            'learning_rate': DET_LEARNING_RATE,
            'warmup_epoch': 5,
        },
        'regularizer': {
            'name': 'L2',
            'factor': 5.0e-5,
        },
    },
    'Architecture': {
        'model_type': 'det',
        'algorithm': 'DB',
        'Transform': None,
        'Backbone': {
            'name': 'PPLCNetV3',
            'scale': 0.75,
            'det': True,
        },
        'Neck': {
            'name': 'RSEFPN',
            'out_channels': 96,
            'shortcut': True,
        },
        'Head': {
            'name': 'DBHead',
            'k': 50,
        },
    },
    'Loss': {
        'name': 'DBLoss',
        'balance_loss': True,
        'main_loss_type': 'DiceLoss',
        'alpha': 5,
        'beta': 10,
        'ohem_ratio': 3,
    },
    'PostProcess': {
        'name': 'DBPostProcess',
        'thresh': 0.3,
        'box_thresh': 0.6,
        'max_candidates': 1000,
        'unclip_ratio': 1.5,
    },
    'Metric': {
        'name': 'DetMetric',
        'main_indicator': 'hmean',
    },
    'Train': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(det_data_dir),
            'label_file_list': [str(det_data_dir / 'train_label.txt')],
            'ratio_list': [1.0],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'DetLabelEncode': None},
                {'IaaAugment': {
                    'augmenter_args': [
                        {'type': 'Fliplr', 'args': {'p': 0.5}},
                        {'type': 'Affine', 'args': {'rotate': [-10, 10]}},
                        {'type': 'Resize', 'args': {'size': [0.5, 3]}},
                    ]
                }},
                {'EastRandomCropData': {
                    'size': [960, 960],
                    'max_tries': 50,
                    'keep_ratio': True,
                }},
                {'MakeBorderMap': {
                    'shrink_ratio': 0.4,
                    'thresh_min': 0.3,
                    'thresh_max': 0.7,
                }},
                {'MakeShrinkMap': {
                    'shrink_ratio': 0.4,
                    'min_text_size': 8,
                }},
                {'NormalizeImage': {
                    'scale': '1./255.',
                    'mean': [0.485, 0.456, 0.406],
                    'std': [0.229, 0.224, 0.225],
                    'order': 'hwc',
                }},
                {'ToCHWImage': None},
                {'KeepKeys': {'keep_keys': ['image', 'threshold_map', 'threshold_mask', 'shrink_map', 'shrink_mask']}},
            ],
        },
        'loader': {
            'shuffle': True,
            'drop_last': False,
            'batch_size_per_card': DET_BATCH_SIZE,
            'num_workers': 4,
        },
    },
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(det_data_dir),
            'label_file_list': [str(det_data_dir / 'val_label.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'DetLabelEncode': None},
                {'DetResizeForTest': {'limit_side_len': 960, 'limit_type': 'max'}},
                {'NormalizeImage': {
                    'scale': '1./255.',
                    'mean': [0.485, 0.456, 0.406],
                    'std': [0.229, 0.224, 0.225],
                    'order': 'hwc',
                }},
                {'ToCHWImage': None},
                {'KeepKeys': {'keep_keys': ['image', 'shape', 'polys', 'ignore_tags']}},
            ],
        },
        'loader': {
            'shuffle': False,
            'drop_last': False,
            'batch_size_per_card': 1,
            'num_workers': 2,
        },
    },
}

det_config_path = DET_OUTPUT_DIR / 'det_prescription_train.yml'
with open(det_config_path, 'w') as f:
    yaml.dump(det_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Detection config saved to {det_config_path}")
print(f"  Epochs: {DET_EPOCHS}, Batch size: {DET_BATCH_SIZE}, LR: {DET_LEARNING_RATE}")
print(f"  Pretrained: {det_pretrained_path}")
print(f"  Training on {len(det_train_labels)} full-page images")

In [ ]:
# Train detection model on full prescription pages!
!cd {WORK_DIR}/PaddleOCR && python tools/train.py -c {det_config_path}

## 6. Train Recognition Model (SVTR_LCNet)

Fine-tunes the recognition model on text crops extracted from the full prescription pages.

In [ ]:
REC_EPOCHS = 50
REC_BATCH_SIZE = 64
REC_LEARNING_RATE = 0.0005

REC_OUTPUT_DIR = Path(WORK_DIR) / 'model' / 'rec'
REC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(dict_file, 'r') as f:
    num_chars = len(f.read().strip().split('\n'))
num_classes = num_chars + 2

rec_config = {
    'Global': {
        'debug': False,
        'use_gpu': True,
        'epoch_num': REC_EPOCHS,
        'log_smooth_window': 20,
        'print_batch_step': 10,
        'save_model_dir': str(REC_OUTPUT_DIR / 'training_output'),
        'save_epoch_step': 10,
        'eval_batch_step': [0, 200],
        'cal_metric_during_train': True,
        'pretrained_model': rec_pretrained_path,
        'checkpoints': None,
        'save_inference_dir': str(REC_OUTPUT_DIR / 'inference'),
        'use_visualdl': False,
        'infer_img': None,
        'character_dict_path': str(dict_file),
        'max_text_length': 80,
        'infer_mode': False,
        'use_space_char': True,
        'distributed': False,
        'save_res_path': str(REC_OUTPUT_DIR / 'rec_results.txt'),
    },
    'Optimizer': {
        'name': 'Adam',
        'beta1': 0.9,
        'beta2': 0.999,
        'lr': {
            'name': 'Cosine',
            'learning_rate': REC_LEARNING_RATE,
            'warmup_epoch': 5,
        },
        'regularizer': {
            'name': 'L2',
            'factor': 3.0e-5,
        },
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'SVTR_LCNet',
        'Transform': None,
        'Backbone': {
            'name': 'PPLCNetV3',
            'scale': 0.95,
        },
        'Head': {
            'name': 'MultiHead',
            'head_list': [
                {
                    'CTCHead': {
                        'Neck': {
                            'name': 'svtr',
                            'dims': 120,
                            'depth': 2,
                            'hidden_dims': 120,
                            'kernel_size': [1, 3],
                            'use_guide': True,
                        },
                        'Head': {
                            'fc_decay': 1.0e-5,
                        },
                    }
                },
                {
                    'NRTRHead': {
                        'nrtr_dim': 384,
                        'max_text_length': 80,
                    }
                },
            ],
            'out_channels_list': {
                'CTCLabelDecode': num_classes,
                'NRTRLabelDecode': num_classes,
            },
        },
    },
    'Loss': {
        'name': 'MultiLoss',
        'loss_config_list': [
            {'CTCLoss': None},
            {'NRTRLoss': None},
        ],
    },
    'PostProcess': {
        'name': 'CTCLabelDecode',
    },
    'Metric': {
        'name': 'RecMetric',
        'main_indicator': 'acc',
        'ignore_space': False,
    },
    'Train': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(rec_data_dir),
            'ext_op_transform_idx': 1,
            'label_file_list': [str(rec_data_dir / 'train_label.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'RecConAug': {
                    'prob': 0.5,
                    'ext_data_num': 2,
                    'image_shape': [48, 320, 3],
                    'max_text_length': 80,
                }},
                {'RecAug': None},
                {'MultiLabelEncode': {
                    'gtc_encode': 'NRTRLabelEncode',
                }},
                {'RecResizeImg': {'image_shape': [3, 48, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
            ],
        },
        'loader': {
            'shuffle': True,
            'batch_size_per_card': REC_BATCH_SIZE,
            'drop_last': True,
            'num_workers': 4,
        },
    },
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(rec_data_dir),
            'label_file_list': [str(rec_data_dir / 'val_label.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'MultiLabelEncode': {
                    'gtc_encode': 'NRTRLabelEncode',
                }},
                {'RecResizeImg': {'image_shape': [3, 48, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
            ],
        },
        'loader': {
            'shuffle': False,
            'drop_last': False,
            'batch_size_per_card': REC_BATCH_SIZE,
            'num_workers': 4,
        },
    },
}

rec_config_path = REC_OUTPUT_DIR / 'rec_prescription_train.yml'
with open(rec_config_path, 'w') as f:
    yaml.dump(rec_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Recognition config saved to {rec_config_path}")
print(f"  Epochs: {REC_EPOCHS}, Batch size: {REC_BATCH_SIZE}, LR: {REC_LEARNING_RATE}")
print(f"  Pretrained: {rec_pretrained_path}")
print(f"  Dictionary: {num_chars} chars, Classes: {num_classes}")
print(f"  Training on {len(rec_train_samples)} text crops")

In [ ]:
# Train recognition model!
!cd {WORK_DIR}/PaddleOCR && python tools/train.py -c {rec_config_path}

## 7. Export Both Models for Inference

In [ ]:
import glob

def find_best_checkpoint(output_dir, work_dir):
    """Search for the best trained checkpoint"""
    training_output = output_dir / 'training_output'
    alt_output = Path(work_dir) / 'PaddleOCR' / 'output'

    print(f"Looking for checkpoints in: {training_output}")
    if training_output.exists():
        for f in sorted(training_output.iterdir()):
            print(f"  {f.name}")

    for search_dir in [training_output, alt_output]:
        if not search_dir.exists():
            continue
        best = search_dir / 'best_accuracy'
        latest = search_dir / 'latest'
        if Path(str(best) + '.pdparams').exists():
            print(f"Using best_accuracy from {search_dir}")
            return str(best)
        elif Path(str(latest) + '.pdparams').exists():
            print(f"Using latest from {search_dir}")
            return str(latest)

    found = glob.glob(str(Path(work_dir) / '**' / 'best_accuracy.pdparams'), recursive=True)
    if not found:
        found = glob.glob(str(Path(work_dir) / '**' / 'latest.pdparams'), recursive=True)
    if found:
        checkpoint = found[0].replace('.pdparams', '')
        print(f"Found checkpoint via search: {checkpoint}")
        return checkpoint

    raise FileNotFoundError(f"No checkpoint found in {training_output} or {alt_output}")


print("=== Detection Model ===")
det_checkpoint = find_best_checkpoint(DET_OUTPUT_DIR, WORK_DIR)

print("\n=== Recognition Model ===")
rec_checkpoint = find_best_checkpoint(REC_OUTPUT_DIR, WORK_DIR)

In [ ]:
import subprocess
import shutil

# Export detection model
det_inference_dir = DET_OUTPUT_DIR / 'inference'
print("Exporting DETECTION model...")
det_export_cmd = (
    f"cd {WORK_DIR}/PaddleOCR && python tools/export_model.py "
    f"-c {det_config_path} "
    f'-o Global.pretrained_model="{det_checkpoint}" '
    f'   Global.save_inference_dir="{det_inference_dir}"'
)
result = subprocess.run(det_export_cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

if not (det_inference_dir.exists() and any(det_inference_dir.glob('*.pdmodel'))):
    found = glob.glob(str(Path(WORK_DIR) / '**' / 'inference.pdmodel'), recursive=True)
    for f in found:
        parent = Path(f).parent
        if 'det' in str(parent).lower():
            print(f"Moving from {parent} to {det_inference_dir}")
            det_inference_dir.mkdir(parents=True, exist_ok=True)
            shutil.copytree(str(parent), str(det_inference_dir), dirs_exist_ok=True)
            break

if det_inference_dir.exists() and any(det_inference_dir.glob('*.pdmodel')):
    print(f"Detection export SUCCESS: {det_inference_dir}")
    for f in sorted(det_inference_dir.iterdir()):
        print(f"  {f.name}: {f.stat().st_size / (1024*1024):.1f} MB")
else:
    print("Detection export FAILED!")

In [ ]:
# Export recognition model
rec_inference_dir = REC_OUTPUT_DIR / 'inference'
print("Exporting RECOGNITION model...")
rec_export_cmd = (
    f"cd {WORK_DIR}/PaddleOCR && python tools/export_model.py "
    f"-c {rec_config_path} "
    f'-o Global.pretrained_model="{rec_checkpoint}" '
    f'   Global.save_inference_dir="{rec_inference_dir}"'
)
result = subprocess.run(rec_export_cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

if not (rec_inference_dir.exists() and any(rec_inference_dir.glob('*.pdmodel'))):
    found = glob.glob(str(Path(WORK_DIR) / '**' / 'inference.pdmodel'), recursive=True)
    for f in found:
        parent = Path(f).parent
        if 'rec' in str(parent).lower():
            print(f"Moving from {parent} to {rec_inference_dir}")
            rec_inference_dir.mkdir(parents=True, exist_ok=True)
            shutil.copytree(str(parent), str(rec_inference_dir), dirs_exist_ok=True)
            break

if rec_inference_dir.exists() and any(rec_inference_dir.glob('*.pdmodel')):
    print(f"Recognition export SUCCESS: {rec_inference_dir}")
    for f in sorted(rec_inference_dir.iterdir()):
        print(f"  {f.name}: {f.stat().st_size / (1024*1024):.1f} MB")
else:
    print("Recognition export FAILED!")

## 8. End-to-End Test on Full Prescription Pages

In [ ]:
from paddleocr import PaddleOCR
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    show_log=False,
    det_model_dir=str(det_inference_dir),
    rec_model_dir=str(rec_inference_dir),
    rec_char_dict_path=str(dict_file),
)

val_images = sorted(det_val_dir.glob('*.png'))[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 20))
axes = axes.flatten()

for i, img_path in enumerate(val_images):
    result = ocr.ocr(str(img_path), cls=True)
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(img_path.name, fontsize=9)

    if result and result[0]:
        for item in result[0]:
            bbox, (text, conf) = item
            x1 = min(p[0] for p in bbox)
            y1 = min(p[1] for p in bbox)
            x2 = max(p[0] for p in bbox)
            y2 = max(p[1] for p in bbox)
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=1, edgecolor='green', facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-2, f'{text} [{conf:.2f}]',
                        fontsize=5, color='green', backgroundcolor='white')
    axes[i].axis('off')

plt.suptitle('End-to-End: Custom Detection + Custom Recognition', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print full OCR output for one sample
sample = val_images[0] if val_images else None
if sample:
    result = ocr.ocr(str(sample), cls=True)
    print(f"=== OCR Result for {sample.name} ===")
    if result and result[0]:
        items = sorted(result[0], key=lambda r: r[0][0][1])
        for item in items:
            bbox, (text, conf) = item
            print(f"  [{conf:.3f}] {text}")
        print(f"\nTotal lines detected: {len(items)}")
        avg_conf = np.mean([item[1][1] for item in items])
        print(f"Average confidence: {avg_conf:.3f}")
    else:
        print("  No text detected!")

## 9. Download Trained Models

**Place the downloaded files in your project:**
```
ml-models/prescription-ocr/model/
    det_inference/          <-- detection model
        inference.pdmodel
        inference.pdiparams
        inference.pdiparams.info
    inference/              <-- recognition model
        inference.pdmodel
        inference.pdiparams
        inference.pdiparams.info
ml-models/prescription-ocr/training_data/
    en_dict.txt
```

In [ ]:
import shutil

export_dir = Path(WORK_DIR) / 'export_for_local'
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(exist_ok=True)

shutil.copytree(det_inference_dir, export_dir / 'det_inference')
shutil.copytree(rec_inference_dir, export_dir / 'inference')
shutil.copy(dict_file, export_dir / 'en_dict.txt')

zip_path = Path(WORK_DIR) / 'trained_fullpage_prescription_model'
shutil.make_archive(str(zip_path), 'zip', str(export_dir))
print(f"Zip created: {zip_path}.zip")
print(f"Size: {Path(str(zip_path) + '.zip').stat().st_size / (1024*1024):.1f} MB")

print("\nContents:")
for root, dirs, files in os.walk(export_dir):
    level = root.replace(str(export_dir), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size = (Path(root) / f).stat().st_size / (1024*1024)
        print(f"{indent}  {f} ({size:.1f} MB)")

In [ ]:
from google.colab import files
files.download(str(zip_path) + '.zip')

## 10. After Download -- Local Setup

```
1. Extract trained_fullpage_prescription_model.zip
2. Copy det_inference/ -> ml-models/prescription-ocr/model/det_inference/
3. Copy inference/     -> ml-models/prescription-ocr/model/inference/
4. Copy en_dict.txt    -> ml-models/prescription-ocr/training_data/en_dict.txt
5. Start the API: python api_service.py
   -> It will auto-detect BOTH custom models and load them!
```

The updated `api_service.py` will:
- Check `model/det_inference/` for custom detection model
- Check `model/inference/` for custom recognition model
- If both found: `modelType: 'paddleocr-custom-fullpage'`
- If only rec found: `modelType: 'paddleocr-custom'`
- If none: `modelType: 'paddleocr-pretrained'`